In [2]:
## Create a vector of the required packages for this analysis
req_packages <- c("janitor", "tidyverse")

## load the packages, quietly
invisible(suppressWarnings(suppressMessages(
    lapply(req_packages, require, character.only = TRUE)
)))

## Create read file

In [16]:
## load in gtf
gtf_raw <- read_tsv("/home/labshare/data/genomes/virilis_group/americana/SB.02.06/Muller/Polished/vir_GCF_030788295.1_liftoff_DameSBpolished.gtf", col_names = FALSE)

## clean up GTF for downstream use
colnames(gtf_raw) <- c("chromosome", "source", "feature", "start", "end", "score", "strand", "frame", "attribute")
gtf <- gtf_raw %>%
    mutate(attribute = str_remove_all(attribute, "\""),
           attribute = str_remove_all(attribute, " ")) %>%
    separate_wider_delim(attribute, delim = ";", names = c("transcript", "gene_id", "gene", "extra"), too_few = "align_start") %>%
    mutate(transcript = str_remove_all(transcript, "transcript_id"),
           gene_id = str_remove_all(gene_id, "gene_id"),
           gene = str_remove_all(gene, "gene_name"),
           chromosome = str_remove_all(chromosome, "_RagTag")) %>%
    filter(feature == "transcript" & grepl("Chr", chromosome)) %>%
    select(chromosome, start, end, gene) %>%
    group_by(chromosome, gene) %>%
    reframe(start = min(start),
            end = max(end))

## load in read information
reads <- read_tsv("../../../analysis/intermediate/sb_dagrp_normalized_reads.tsv")

head(gtf)
head(reads)

Rows: 339314 Columns: 9
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (7): X1, X2, X3, X6, X7, X8, X9
dbl (2): X4, X5

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 10625 Columns: 102
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr   (1): gene
dbl (101): BB_05_18_rep1, BB_05_18_rep2, BB_05_18_rep3, BU_06_06_rep1, BU_06...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


chromosome,gene,start,end
<chr>,<chr>,<dbl>,<dbl>
Chr_2,14-3-3epsilon,4148838,4154854
Chr_2,140up,5550466,5551562
Chr_2,2mit,17050377,17082004
Chr_2,5-HT2A,6298074,6332975
Chr_2,5PtaseI,39985010,40000302
Chr_2,7B2,35210027,35211282


gene,BB_05_18_rep1,BB_05_18_rep2,BB_05_18_rep3,BU_06_06_rep1,BU_06_06_rep2,BU_06_06_rep3,BU_06_10_rep1,BU_06_10_rep2,BU_06_10_rep3,⋯,SV_07_02_rep3,WR_06_22_rep1,WR_06_22_rep2,WR_06_22_rep3,WR_06_44_rep1,WR_06_44_rep2,WR_06_44_rep3,WS_07_06_rep1,WS_07_06_rep2,WS_07_06_rep3
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
cdi,8583,8785,8738,6625,6595,6659,6446,6552,6328,⋯,7384,7539,7085,7024,7990,7339,7546,7000,7394,7385
mRpL55,214,228,256,254,222,261,285,311,336,⋯,306,276,294,267,405,427,405,354,346,381
ATPsynD,2488,2542,2837,2759,2516,2599,3015,2770,2933,⋯,3673,3335,3295,3564,3137,3587,3170,3656,3471,3797
sav,1203,1218,1218,775,741,772,839,806,971,⋯,1023,887,924,895,870,898,926,1261,1240,1205
Ctns,414,400,448,417,427,387,688,679,602,⋯,308,450,395,447,278,347,293,503,568,469
p53,265,251,266,324,311,313,393,389,397,⋯,243,362,277,310,385,336,391,325,366,306


In [18]:
## calculate the average expression for each strain
avg_expression <- reads %>%
    pivot_longer(!gene, names_to = "sample", values_to = "expression") %>%
    separate_wider_delim(sample, delim = "_", names = c("locality", "female", "number", "replicate"), too_few = "align_start") %>%
    mutate(strain = case_when(is.na(replicate) ~ paste(locality, female, sep = "_"),
                              TRUE ~ paste(locality, female, number, sep = "_"))) %>%
    group_by(gene, strain) %>%
    reframe(avg_expression = mean(expression)) %>%
    pivot_wider(names_from = strain, values_from = avg_expression)

## add gene position information to dataframe
phenotype_df <- gtf %>%
    inner_join(avg_expression, by = "gene") %>%
    rename(chr = chromosome, phenotype_id = gene)
head(phenotype_df)

chr,phenotype_id,start,end,BB_05_18,BU_06_06,BU_06_10,CB_05_22,CD_97_5,CF_05_14,⋯,PG_05_04,PG_06_36,PM_99_18,RB_10_16,SB_02_06,SC_07_06,SV_07_02,WR_06_22,WR_06_44,WS_07_06
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Chr_2,14-3-3epsilon,4148838,4154854,14109.0000,13590.66667,15980.66667,15434.66667,15254.66667,16027.33333,⋯,15054.33333,16707.0000,17037.66667,16327.66667,14025.00000,15243.00000,14022.000,14764.0000,15842.33333,14383.66667
Chr_2,140up,5550466,5551562,694.3333,1171.00000,912.66667,792.66667,1011.00000,1067.33333,⋯,945.66667,946.3333,810.33333,733.00000,1026.66667,736.66667,1017.333,972.6667,686.33333,1018.33333
Chr_2,2mit,17050377,17082004,43.0000,51.33333,113.00000,45.00000,186.33333,91.00000,⋯,69.33333,106.6667,47.66667,66.33333,34.66667,29.66667,65.000,71.0000,45.33333,26.00000
Chr_2,5-HT2A,6298074,6332975,1924.6667,1462.33333,1888.66667,1545.33333,1250.66667,1407.66667,⋯,1919.33333,1886.3333,1683.33333,1776.33333,1749.66667,1879.66667,1358.333,1476.0000,1884.66667,1650.00000
Chr_2,5PtaseI,39985010,40000302,773.6667,549.33333,778.00000,821.66667,729.66667,741.33333,⋯,801.00000,714.0000,734.33333,823.33333,634.33333,755.66667,700.000,681.0000,941.33333,648.66667
Chr_2,7B2,35210027,35211282,51.0000,160.00000,82.66667,98.66667,78.66667,63.33333,⋯,145.66667,154.3333,99.66667,99.66667,77.33333,70.00000,84.000,76.0000,84.00000,86.33333


## Create covariate file

In [19]:
## load in metadata
meta <- read_csv("../../../analysis/input/YABLab_Drosophila_stocks.csv") %>%
    janitor::clean_names() %>%
    mutate(strain_id = str_replace_all(strain_id, "\\.", "_"))
head(meta)

Rows: 294 Columns: 21
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (16): species_group, Species, strain_ID, strain_type, Genotype, label, s...
dbl  (5): year, latitude, longitude, water_distance_mi, riparian_year

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


species_group,species,strain_id,strain_type,genotype,label,year,slot,saved_on_side,locality,⋯,state,country,latitude,longitude,sequencing,wetland,water_distance_mi,water_source,riparian,riparian_year
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>
virilis,D. americana,15010-0951_00,wild-type,wt,Dame\wild-type,NA,A1-1,NA,"Anderson, Indiana.",⋯,IN,USA,NA,NA,BernardKim_nanopore,NA,NA,NA,NA,NA
virilis,D. americana,15010-0951_01,wild-type,wt,Dame\wild-type,1947,A1-2,NA,"Poplar, Montana.",⋯,MT,USA,NA,NA,NA,NA,NA,NA,NA,NA
virilis,D. americana,15010-0951_02,wild-type,wt,Dame\wild-type,1947,A1-3,NA,"Chinook, Montana.",⋯,MT,USA,NA,NA,NA,NA,NA,NA,NA,NA
virilis,D. americana,15010-0951_03,wild-type,wt,Dame\wild-type,1948,A1-4,NA,"Millersburg, Pennsylvania.",⋯,PA,USA,NA,NA,NA,NA,NA,NA,NA,NA
virilis,D. americana,15010-0951_04,wild-type,wt,Dame\wild-type,1948,A1-5,NA,"Keelers Bay, Lake Champlain, Vermont",⋯,VT,USA,NA,NA,NA,NA,NA,NA,NA,NA
virilis,D. americana,15010-0951_05,wild-type,wt,Dame\wild-type,1948,A2-1,NA,"Jackson, Michigan.",⋯,MI,USA,NA,NA,NA,NA,NA,NA,NA,NA


In [20]:
## get list of strains with expression data
strains <- colnames(phenotype_df)[-c(1,2,3,4)]

## filter metadata for only the strains of interest
covariates <- meta %>%
    select(strain_id, latitude) %>%
    filter(strain_id %in% strains) %>%
    na.omit() %>%
    separate_wider_delim(strain_id, delim = "_", names = c("locality", "female", "number"), too_few = "align_start", cols_remove = FALSE) %>%
    select(strain_id, locality, latitude) %>%
    mutate(locality = as.numeric(factor(locality))) %>%
    t() %>%
    as.data.frame() %>%
    row_to_names(1)  %>%
    rownames_to_column("variables")
head(covariates)

,variables,BB_05_18,BU_06_06,BU_06_10,CB_05_22,CI_05_20,DA_06_36,DI_10_02,DN_07_B,FG_06_22,⋯,OR_01_34,PG_05_04,PM_99_18,RB_10_16,SB_02_06,SC_07_06,SV_07_02,WR_06_22,WR_06_44,WS_07_06
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,locality,1,2,2,3,4,5,6,7,8,⋯,18,19,20,21,22,23,24,25,25,26
2,latitude,36.46119,31.99426,31.99426,32.98000,30.75824,32.54012,40.45987,41.36827,30.70849,⋯,41.61621,44.09362,36.97485,32.53755,41.49610,40.11310,39.61408,34.59891,34.59891,40.71180


In [21]:
## save phenotype dataframe as a BED file
write_tsv(covariates, "input/covariates.txt")

In [25]:
## filter phenotype data for only the strains with latitude information
select_phenotype_df <- phenotype_df %>%
    select(chr, start, end, phenotype_id, any_of(colnames(covariates))) %>%
    arrange(chr, start)
head(select_phenotype_df)

chr,start,end,phenotype_id,BB_05_18,BU_06_06,BU_06_10,CB_05_22,CI_05_20,DA_06_36,⋯,OR_01_34,PG_05_04,PM_99_18,RB_10_16,SB_02_06,SC_07_06,SV_07_02,WR_06_22,WR_06_44,WS_07_06
<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Chr_2,94624,112431,cdi,8702.0000,6626.3333,6442.0000,8039.6667,7515.6667,5494.6667,⋯,7326.6667,3160.3333,6703.3333,6965.0000,6912.6667,7490.6667,7250.0000,7216.0000,7625.0000,7259.6667
Chr_2,112589,113113,mRpL55,232.6667,245.6667,310.6667,248.6667,312.0000,261.3333,⋯,320.6667,219.6667,310.6667,359.6667,381.3333,320.0000,289.6667,279.0000,412.3333,360.3333
Chr_2,113108,114233,ATPsynD,2622.3333,2624.6667,2906.0000,2881.0000,3287.6667,2805.0000,⋯,2855.3333,2370.0000,3521.0000,2986.0000,3656.3333,2954.3333,3717.6667,3398.0000,3298.0000,3641.3333
Chr_2,114638,117503,sav,1213.0000,762.6667,872.0000,1014.0000,1019.0000,1131.0000,⋯,1082.0000,948.6667,919.6667,1102.6667,1166.3333,1038.3333,1003.3333,902.0000,898.0000,1235.3333
Chr_2,119251,121213,Ctns,420.6667,410.3333,656.3333,448.6667,293.3333,402.3333,⋯,579.6667,302.0000,400.6667,629.6667,386.3333,230.6667,364.3333,430.6667,306.0000,513.3333
Chr_2,121842,125702,p53,260.6667,316.0000,393.0000,325.0000,277.6667,351.3333,⋯,320.0000,303.3333,282.6667,430.0000,296.0000,353.0000,272.6667,316.3333,370.6667,332.3333


In [34]:
## identify gene(s) that are repeated
unique(select_phenotype_df$phenotype_id[duplicated(select_phenotype_df$phenotype_id)])

[1] "TRNAY-GUA"

In [36]:
## create unique name for the one gene that 
select_phenotype_df <- select_phenotype_df %>%
    mutate(phenotype_id = case_when(phenotype_id == "TRNAY-GUA" ~ paste(phenotype_id, chr, sep = "_"),
                                    TRUE ~ phenotype_id))
head(select_phenotype_df)

chr,start,end,phenotype_id,BB_05_18,BU_06_06,BU_06_10,CB_05_22,CI_05_20,DA_06_36,⋯,OR_01_34,PG_05_04,PM_99_18,RB_10_16,SB_02_06,SC_07_06,SV_07_02,WR_06_22,WR_06_44,WS_07_06
<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Chr_2,94624,112431,cdi,8702.0000,6626.3333,6442.0000,8039.6667,7515.6667,5494.6667,⋯,7326.6667,3160.3333,6703.3333,6965.0000,6912.6667,7490.6667,7250.0000,7216.0000,7625.0000,7259.6667
Chr_2,112589,113113,mRpL55,232.6667,245.6667,310.6667,248.6667,312.0000,261.3333,⋯,320.6667,219.6667,310.6667,359.6667,381.3333,320.0000,289.6667,279.0000,412.3333,360.3333
Chr_2,113108,114233,ATPsynD,2622.3333,2624.6667,2906.0000,2881.0000,3287.6667,2805.0000,⋯,2855.3333,2370.0000,3521.0000,2986.0000,3656.3333,2954.3333,3717.6667,3398.0000,3298.0000,3641.3333
Chr_2,114638,117503,sav,1213.0000,762.6667,872.0000,1014.0000,1019.0000,1131.0000,⋯,1082.0000,948.6667,919.6667,1102.6667,1166.3333,1038.3333,1003.3333,902.0000,898.0000,1235.3333
Chr_2,119251,121213,Ctns,420.6667,410.3333,656.3333,448.6667,293.3333,402.3333,⋯,579.6667,302.0000,400.6667,629.6667,386.3333,230.6667,364.3333,430.6667,306.0000,513.3333
Chr_2,121842,125702,p53,260.6667,316.0000,393.0000,325.0000,277.6667,351.3333,⋯,320.0000,303.3333,282.6667,430.0000,296.0000,353.0000,272.6667,316.3333,370.6667,332.3333


In [37]:
## save phenotype dataframe as a BED file
write_tsv(select_phenotype_df, "input/phenotype.bed")

In [27]:
## get list of samples as a data frame
colnames(covariates)[-1] %>%
    as.data.frame() %>%
    write_tsv("input/samples.tsv", col_names = FALSE)